In [1]:
# !pip install growwapi
# !pip install --upgrade growwapi
# !pip install pyarrow fastparquet
# !pip install --upgrade scipy statsmodels
# !pip install scipy==1.10.1 statsmodels==0.14.0

In [2]:
# DATA:
# → Store → Merge → Clean

# SPLIT:
# → Train / Test

# TRAIN (Selection Phase):
# → Correlation (prices)
# → Cointegration
# → Static Hedge Ratio
# → Spread
# → Half-life
# → Select Pairs

# TEST (Trading Phase):
# → Rolling Hedge Ratio
# → Spread
# → Rolling Z-score
# → Signals
# → Positions
# → Returns
# → PnL
# → Portfolio Metrics

In [3]:
import pandas as pd
import numpy as np
import math
from itertools import combinations
import os
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import statsmodels.api as sm
from statsmodels.tsa.stattools import coint
from growwapi import GrowwAPI

In [4]:
API_Key = "eyJraWQiOiJaTUtjVXciLCJhbGciOiJFUzI1NiJ9.eyJleHAiOjI1NjM4NDU1MTQsImlhdCI6MTc3NTQ0NTUxNCwibmJmIjoxNzc1NDQ1NTE0LCJzdWIiOiJ7XCJ0b2tlblJlZklkXCI6XCI1YThlOWU5MC02MGNjLTQ1NDUtODY1Yi0yZmI0MDdhODRhMTNcIixcInZlbmRvckludGVncmF0aW9uS2V5XCI6XCJlMzFmZjIzYjA4NmI0MDZjODg3NGIyZjZkODQ5NTMxM1wiLFwidXNlckFjY291bnRJZFwiOlwiODdlODE1ZjAtZWM5MS00OTU4LWIwNzUtMDg0MWMyMTk2MzNkXCIsXCJkZXZpY2VJZFwiOlwiNTEyN2ZiZjktMTZiMS01NzUzLTk2MjgtOTQyMTQwNGJhZDYwXCIsXCJzZXNzaW9uSWRcIjpcIjlhZjlkYTc1LWFkNGQtNGM3NS1hMjZjLWRkMTc0OThhYTc5MVwiLFwiYWRkaXRpb25hbERhdGFcIjpcIno1NC9NZzltdjE2WXdmb0gvS0EwYkFvc2NJMXRRNzBlTXdSeHhOa1IxMTlSTkczdTlLa2pWZDNoWjU1ZStNZERhWXBOVi9UOUxIRmtQejFFQisybTdRPT1cIixcInJvbGVcIjpcImF1dGgtdG90cFwiLFwic291cmNlSXBBZGRyZXNzXCI6XCI0OS4yNDguMTAyLjE1OCwxNzIuNzEuMTk4Ljk0LDM1LjI0MS4yMy4xMjNcIixcInR3b0ZhRXhwaXJ5VHNcIjoyNTYzODQ1NTE0NzI0LFwidmVuZG9yTmFtZVwiOlwiZ3Jvd3dBcGlcIn0iLCJpc3MiOiJhcGV4LWF1dGgtcHJvZC1hcHAifQ.bgjJsha_GiSnbd3cu0Gl9chkWYJyD_qhlEJyz1ujdZbxFOHfrnE8yY0NEFYlYuKoK4f4fOGoHt1_zbiu4O1n6Q"
Secret_Key = "g0VR3jBal&rZld$s%faywsSLJsDcCj$R"

In [5]:
access_token = GrowwAPI.get_access_token(api_key=API_Key, secret=Secret_Key)
# Use access_token to initiate GrowwAPI
groww = GrowwAPI(access_token)

Ready to Groww!


In [6]:
symbol_segment_map = {'RELIANCE': 'Oil Gas & Consumable Fuels', 'HDFCBANK': 'Financial Services', 
                      'ICICIBANK': 'Financial Services', 'INFY': 'Information Technology', 'TCS': 'Information Technology',
                      'HINDUNILVR': 'FMCG', 'ITC': 'FMCG', 'BHARTIARTL': 'Telecommunication', 'SBIN': 'Financial Services',
                      'KOTAKBANK': 'Financial Services', 'AXISBANK': 'Financial Services', 'BAJFINANCE': 'Financial Services',
                      'BAJAJFINSV': 'Financial Services', 'HCLTECH': 'Information Technology', 
                      'WIPRO': 'Information Technology', 'LTIM': 'Information Technology', 'TECHM': 'Information Technology',
                      'LT': 'Construction', 'ULTRACEMCO': 'Construction Materials', 'GRASIM': 'Construction Materials', 
                      'SHREECEM': 'Construction Materials', 'TATASTEEL': 'Metals & Mining', 'JSWSTEEL': 'Metals & Mining', 
                      'HINDALCO': 'Metals & Mining', 'COALINDIA': 'Oil Gas & Consumable Fuels', 
                      'ONGC': 'Oil Gas & Consumable Fuels', 'POWERGRID': 'Power', 'NTPC': 'Power', 
                      'ADANIENT': 'Diversified', 'ADANIPORTS': 'Services', 'ASIANPAINT': 'Consumer Durables', 
                      'TITAN': 'Consumer Durables', 'NESTLEIND': 'FMCG', 'BRITANNIA': 'FMCG', 'MARUTI': 'Automobile', 
                      'M&M': 'Automobile', 'TATAMOTORS': 'Automobile', 'BAJAJ-AUTO': 'Automobile', 
                      'HEROMOTOCO': 'Automobile', 'EICHERMOT': 'Automobile', 'APOLLOHOSP': 'Healthcare', 
                      'SUNPHARMA': 'Healthcare', 'DRREDDY': 'Healthcare', 'CIPLA': 'Healthcare', 'DIVISLAB': 'Healthcare', 
                      'UPL': 'Chemicals', 'SBILIFE': 'Financial Services', 'HDFCLIFE': 'Financial Services', 
                      'SBICARD': 'Financial Services', 'INDUSINDBK': 'Financial Services'}


# -------------------------------------------ETL-------------------------------------------

### Historical Data Storing

In [7]:
def get_historical_data(start_date, end_date, symbol="RELIANCE"):
    historical_data_response = groww.get_historical_candles(
        groww_symbol=f"NSE-{symbol.upper()}",
        exchange=groww.EXCHANGE_NSE,
        segment=groww.SEGMENT_CASH,
        start_time=start_date,
        end_time=end_date,
        candle_interval=groww.CANDLE_INTERVAL_DAY # Optional: Interval in days for the candle data
    )
    return historical_data_response

In [8]:
def fetch_and_store_hist_data(symbol):    
    file_path = os.path.join(os.getcwd(), f"{symbol}_Hist_Data.csv")
    
    if os.path.exists(file_path):
        print(f"Data already exists for {symbol}")
        
    else:
        start_date = pd.to_datetime('2020-01-01')
        end_date = pd.to_datetime('2026-01-01')
        total_days = end_date - start_date
        data_list = []
        while total_days > timedelta(days=0):
            intermediate_end_date = start_date + timedelta(days=180) if total_days > timedelta(days=180) else start_date + timedelta(days=total_days.days)
            print(start_date, " => ", intermediate_end_date)
            hist_data = get_historical_data(start_date, intermediate_end_date, symbol)
            print(hist_data["candles"], '\n\n')
            if hist_data["candles"]:
                data_list.append(hist_data["candles"])
            start_date = intermediate_end_date
            total_days = total_days - timedelta(days=180)
        #     print(total_days, '\n')

        data_list = np.concatenate(data_list)
        data_list

        df = pd.DataFrame(
            data_list,
            columns=["Date", "Open", "High", "Low", "Close", "Volume", "Extra"]
        )

        df.to_csv(f"{symbol}_Hist_Data.csv")

In [9]:
# stock_df = fetch_and_store_hist_data("LTIM")
# stock_df

### Data Merging

In [10]:
df_list = []
for symbol in symbol_segment_map.keys():
    df = pd.read_csv(f"{symbol}_Hist_Data.csv")
    df["Date"] = pd.to_datetime(df["Date"])
    df.set_index("Date", inplace=True)
    
    # remove duplicate dates
    df = df[~df.index.duplicated(keep="first")]
    
    new_df = df["Close"].rename(symbol)
    df_list.append(new_df)


merged_df = pd.concat(df_list, axis=1)
merged_df.to_csv("Merged_Data.csv")

In [11]:
# merged_df

### Data Cleaning

In [12]:
merged_df = pd.read_csv("Merged_Data.csv", index_col="Date", parse_dates=True)
merged_df = merged_df.ffill()
merged_df = merged_df.dropna()

In [13]:
for col in merged_df.columns:
    col_value_count = merged_df[col].notna().sum()
    if col_value_count < 1490:
        print(col, " => ", col_value_count)

RELIANCE  =>  764
HDFCBANK  =>  764
ICICIBANK  =>  764
INFY  =>  764
TCS  =>  764
HINDUNILVR  =>  764
ITC  =>  764
BHARTIARTL  =>  764
SBIN  =>  764
KOTAKBANK  =>  764
AXISBANK  =>  764
BAJFINANCE  =>  764
BAJAJFINSV  =>  764
HCLTECH  =>  764
WIPRO  =>  764
LTIM  =>  764
TECHM  =>  764
LT  =>  764
ULTRACEMCO  =>  764
GRASIM  =>  764
SHREECEM  =>  764
TATASTEEL  =>  764
JSWSTEEL  =>  764
HINDALCO  =>  764
COALINDIA  =>  764
ONGC  =>  764
POWERGRID  =>  764
NTPC  =>  764
ADANIENT  =>  764
ADANIPORTS  =>  764
ASIANPAINT  =>  764
TITAN  =>  764
NESTLEIND  =>  764
BRITANNIA  =>  764
MARUTI  =>  764
M&M  =>  764
TATAMOTORS  =>  764
BAJAJ-AUTO  =>  764
HEROMOTOCO  =>  764
EICHERMOT  =>  764
APOLLOHOSP  =>  764
SUNPHARMA  =>  764
DRREDDY  =>  764
CIPLA  =>  764
DIVISLAB  =>  764
UPL  =>  764
SBILIFE  =>  764
HDFCLIFE  =>  764
SBICARD  =>  764
INDUSINDBK  =>  764


In [14]:
# merged_df.drop(["LTIM", "TATAMOTORS", "SBICARD"], axis=1, inplace=True)

In [15]:
merged_df = merged_df.ffill()

In [16]:
train_df = merged_df.loc[:'2023-12-31']
test_df = merged_df.loc['2024-01-01':]

# -------------------------------------------Training Data Analysis--------------------------------------

### Correlation

In [17]:
corr_matrix = train_df.corr()
corr_matrix

,RELIANCE,HDFCBANK,ICICIBANK,INFY,TCS,HINDUNILVR,ITC,BHARTIARTL,SBIN,KOTAKBANK,...,APOLLOHOSP,SUNPHARMA,DRREDDY,CIPLA,DIVISLAB,UPL,SBILIFE,HDFCLIFE,SBICARD,INDUSINDBK
RELIANCE,1.000000,0.529197,0.421881,0.025537,0.072152,0.697102,0.129482,0.125121,0.613280,0.627471,...,0.319430,0.021891,0.023815,0.124479,0.432162,0.105651,0.312426,0.426262,0.552649,0.229718
HDFCBANK,0.529197,1.000000,0.106740,-0.032395,-0.060543,0.432206,-0.048551,-0.299640,0.212714,0.615149,...,-0.150406,-0.282586,-0.239513,-0.357865,-0.088511,0.493826,-0.205332,-0.120639,0.248489,-0.259201
ICICIBANK,0.421881,0.106740,1.000000,-0.187570,0.507140,0.132829,0.771389,0.770932,0.702912,0.467131,...,0.810897,0.667343,0.763340,0.609314,0.816965,-0.656264,0.711707,0.798863,0.509413,0.833589
INFY,0.025537,-0.032395,-0.187570,1.000000,0.576641,-0.066110,-0.451966,0.100772,0.160068,-0.411218,...,-0.025937,0.254342,-0.098627,0.352407,-0.095091,0.131467,0.215028,-0.026243,-0.492180,0.019985
TCS,0.072152,-0.060543,0.507140,0.576641,1.000000,-0.157334,0.357594,0.702005,0.486665,-0.122332,...,0.601058,0.770745,0.570407,0.688869,0.488628,-0.516010,0.666235,0.522493,-0.136498,0.668861
HINDUNILVR,0.697102,0.432206,0.132829,-0.066110,-0.157334,1.000000,-0.033883,-0.082527,0.352171,0.576568,...,0.077632,-0.239981,-0.280622,-0.139541,0.192992,0.305404,0.080763,0.182683,0.546079,-0.000921
ITC,0.129482,-0.048551,0.771389,-0.451966,0.357594,-0.033883,1.000000,0.651619,0.227011,0.234730,...,0.740398,0.541987,0.796310,0.357813,0.607693,-0.731279,0.455630,0.612848,0.456250,0.707191
BHARTIARTL,0.125121,-0.299640,0.770932,0.100772,0.702005,-0.082527,0.651619,1.000000,0.561440,0.002589,...,0.922926,0.907263,0.822614,0.800876,0.780471,-0.824271,0.887556,0.827540,0.049656,0.936003
SBIN,0.613280,0.212714,0.702912,0.160068,0.486665,0.352171,0.227011,0.561440,1.000000,0.505522,...,0.572746,0.469187,0.306485,0.516195,0.687089,-0.267949,0.673215,0.662712,0.347236,0.630546
KOTAKBANK,0.627471,0.615149,0.467131,-0.411218,-0.122332,0.576568,0.234730,0.002589,0.505522,1.000000,...,0.152498,-0.177772,-0.069385,-0.181978,0.280800,0.189498,0.043702,0.178614,0.636726,0.075325


In [18]:
# plt.figure(figsize=(20,16))
# sns.heatmap(corr_matrix)

In [19]:
# corr_matrix.columns

In [20]:
# Create stocks pairs

stocks = merged_df.columns
pairs = list(combinations(stocks, 2))
# pairs

In [21]:
cross_sector_pairs = []
for s1, s2 in pairs:
    if symbol_segment_map[s1] != symbol_segment_map[s2]:
        cross_sector_pairs.append((s1, s2))

# cross_sector_pairs

In [22]:
correlated_pairs = []

for s1, s2 in cross_sector_pairs:
    corr = corr_matrix.loc[s1, s2]
    
    if abs(corr) > 0.3:
        correlated_pairs.append((s1, s2, round(corr, 2)))
    
# correlated_pairs

In [23]:
correlated_pairs_df = pd.DataFrame(correlated_pairs, columns=["Stock1", "Stock2", "Corr_Value"])
correlated_pairs_df.sort_values(by="Corr_Value", ascending=False)

,Stock1,Stock2,Corr_Value
448,LT,NTPC,0.98
727,TITAN,TATAMOTORS,0.96
447,LT,ONGC,0.96
483,ULTRACEMCO,BAJAJ-AUTO,0.95
636,ONGC,NTPC,0.95
...,...,...,...
847,UPL,INDUSINDBK,-0.86
832,DRREDDY,UPL,-0.87
736,TITAN,UPL,-0.87
793,TATAMOTORS,UPL,-0.89


In [24]:
# heatmap_matrix = correlated_pairs_df.pivot(index="Stock1", columns="Stock2", values="Corr_Value")
# heatmap_matrix

In [25]:
# plt.figure(figsize=(10,6))

# sns.heatmap(
#     heatmap_matrix,
#     cmap="coolwarm",
#     annot=True,
#     linewidths=0.5
# )

In [26]:
cointegrated_pairs = []

for s1, s2, corr in correlated_pairs:
    
    coint_pair = train_df[[s1, s2]].copy()
    coint_pair = coint_pair.replace([-np.inf, np.inf], np.nan).dropna()
    score, p_value, _ = coint(merged_df[s1], merged_df[s2])
    
    if len(coint_pair) < 100:
        continue
    
    if p_value < 0.03 and score < -3:
        cointegrated_pairs.append((s1, s2, corr, score, p_value))

# cointegrated_pairs

In [27]:
cointegrated_pairs_df = pd.DataFrame(cointegrated_pairs, columns=["Stock1", "Stock2", "Corr_Score", "CoIn_Score", "P_Value"])
cointegrated_pairs_df.sort_values("P_Value")

,Stock1,Stock2,Corr_Score,CoIn_Score,P_Value
0,BHARTIARTL,JSWSTEEL,0.79,-4.607428,0.000811
10,JSWSTEEL,EICHERMOT,0.60,-4.520441,0.001133
9,JSWSTEEL,M&M,0.85,-4.221721,0.003381
14,BRITANNIA,SBILIFE,0.56,-4.208898,0.003536
6,ULTRACEMCO,APOLLOHOSP,0.89,-4.150381,0.004332
4,BAJAJFINSV,JSWSTEEL,0.79,-4.022571,0.006667
8,TATASTEEL,ADANIPORTS,0.68,-4.020674,0.006709
3,AXISBANK,ADANIPORTS,0.84,-3.960996,0.008156
2,AXISBANK,TATASTEEL,0.82,-3.863717,0.011123
1,SBIN,HINDALCO,0.64,-3.770222,0.014846


### Hedge Ratio

In [28]:
def get_hedge_ratio(y, x):
    x = sm.add_constant(x)
    model = sm.OLS(y, x).fit()
#     print(x, y, model.params, "\n")
    beta = model.params.iloc[1]
    return beta


In [29]:
hedge_ratio = {}
for s1, s2, corr, coin, _ in cointegrated_pairs:
    if not hedge_ratio.get(s1):
        hedge_ratio[s1] = {}
        
    hedge_ratio[s1][s2] = get_hedge_ratio(train_df[s2], train_df[s1])


In [30]:
hedge_ratio

{'BHARTIARTL': {'JSWSTEEL': 0.49618669236312585},
 'SBIN': {'HINDALCO': 0.8822213807856469},
 'AXISBANK': {'TATASTEEL': 0.09913874453990296,
  'ADANIPORTS': 1.3023138375063283},
 'BAJAJFINSV': {'JSWSTEEL': 0.3053129681165867},
 'LT': {'TITAN': 0.8612574634866541},
 'ULTRACEMCO': {'APOLLOHOSP': 0.4613948698150707},
 'GRASIM': {'APOLLOHOSP': 2.465683370177081},
 'TATASTEEL': {'ADANIPORTS': 8.729932521477286},
 'JSWSTEEL': {'M&M': 2.767088319313693, 'EICHERMOT': 3.3547088201766604},
 'ADANIENT': {'ADANIPORTS': 0.10962113451131487, 'CIPLA': 0.07404154477773796},
 'ADANIPORTS': {'HEROMOTOCO': 2.666355948217271},
 'BRITANNIA': {'SBILIFE': 0.22052330531090278}}

### Spread 

In [31]:
spread = {}
for s1, s2, corr, coin, _ in cointegrated_pairs:
    if not spread.get(s1):0
    spread[s1] = {}
    spread[s1][s2] = merged_df[s1] - hedge_ratio[s1][s2] * merged_df[s2]

In [32]:
spread

{'BHARTIARTL': {'JSWSTEEL': Date
  2022-12-05     468.685149
  2022-12-06     462.822587
  2022-12-07     467.739793
  2022-12-08     463.089770
  2022-12-09     467.523373
                   ...     
  2025-12-26    1562.373284
  2025-12-29    1539.466420
  2025-12-30    1548.238873
  2025-12-31    1527.641741
  2026-01-01    1529.117290
  Length: 764, dtype: float64},
 'SBIN': {'HINDALCO': Date
  2022-12-05    192.775072
  2022-12-06    194.305951
  2022-12-07    197.434613
  2022-12-08    195.550286
  2022-12-09    208.163834
                   ...    
  2025-12-26    196.208957
  2025-12-29    201.928506
  2025-12-30    193.433966
  2025-12-31    199.934302
  2026-01-01    195.205975
  Length: 764, dtype: float64},
 'AXISBANK': {'ADANIPORTS': Date
  2022-12-05   -263.961604
  2022-12-06   -262.809240
  2022-12-07   -240.176795
  2022-12-08   -223.486026
  2022-12-09   -226.336051
                   ...    
  2025-12-26   -708.470908
  2025-12-29   -662.085245
  2025-12-30   -656.94

### Half Life

In [ ]:
def calculate_half_life(stock_1, stock_2, beta_val):
    
    spread = merged_df[stock_1] - beta_val * merged_df[stock_2]
    
    spread_lag = spread.shift(1)
    spread_return = spread - spread_lag
    
    spread_lag = spread_lag.dropna()
    spread_return = spread_return.dropna()
    
    spread_lag = sm.add_constant(spread_lag)
    
    model = sm.OLS(spread_return, spread_lag).fit()
    
    beta = model.params.iloc[1]
    
    half_life = -np.log(2) / beta
    
    return half_life
    

In [ ]:
half_lives = []
for _, row in cointegrated_pairs_df.iterrows():
    stock_1 = row["Stock1"]
    stock_2 = row["Stock2"]
    beta_val = hedge_ratio[stock_1][stock_2]
    half_life = calculate_half_life(stock_1, stock_2, beta_val)
    half_lives.append(half_life)

cointegrated_pairs_df["Half_Life"] = half_lives

### Z Score

In [34]:
z_score = {}
for s1, s2, corr, coin, _ in cointegrated_pairs:
    if not z_score.get(s1):
        z_score[s1] = {}
    z_score[s1][s2] = (spread[s1][s2] - spread[s1][s2].mean()) / spread[s1][s2].std()

KeyError: 'TATASTEEL'

In [ ]:
z_score

In [ ]:
# cointegrated_pairs_df[(cointegrated_pairs_df["Stock1"] == "SBIN") & (cointegrated_pairs_df["Stock2"] == "HINDALCO")]["P_Value"].values[0]

In [ ]:
cointegrated_pairs_df.sort_values(by="Half_Life")


In [ ]:
cointegrated_pairs_df

### Rolling Spread and Z-Score

In [ ]:
rolling_z_score = {}
window = 30
for s1, s2, corr, coin, p_value in cointegrated_pairs:
    if not rolling_z_score.get(s1):
        rolling_z_score[s1] = {}
    _spread = spread[s1][s2]
    rolling_mean = _spread.rolling(window).mean()
    rolling_std = _spread.rolling(window).std()
    rolling_z_score[s1][s2] = (_spread - rolling_mean) / rolling_std

In [ ]:
rolling_z_score

In [ ]:
df_list = []

for stock1, inner_dict in rolling_z_score.items():
    for stock2, series in inner_dict.items():
        col_name = f"{stock1}_{stock2}"
        df_list.append(series.rename(col_name))

rolling_z_score_df = pd.concat(df_list, axis=1)

In [ ]:
rolling_z_score_df

### Signal Generation

In [ ]:
entry_threshold = 2
exit_threshold = 0.5
stop_loss = 3

signals_df = pd.DataFrame(index=rolling_z_score_df.index)

for col in rolling_z_score_df.columns:
    
    signals = []
    
    z_series = rolling_z_score_df[col]
    
    for i in range(len(z_series)):
        
        if i == 0 or pd.isna(z_series.iloc[i]):
            signals.append(None)
            continue
        
        prev_z = z_series.iloc[i-1]
        curr_z = z_series.iloc[i]
        
        # LONG ENTRY
        if prev_z < entry_threshold and curr_z > entry_threshold:
            signals.append(1)
        
        # SHORT ENTRY
        elif prev_z > -entry_threshold and curr_z < -entry_threshold:
            signals.append(-1)
        
        # EXIT
        elif abs(curr_z) < exit_threshold:
            signals.append(0)
        
        # STOP LOSS
        elif abs(curr_z) > stop_loss:
            signals.append(0)
        
        else:
            signals.append(None)
    
    signals_df[col] = signals

In [ ]:
signals_df.index = rolling_z_score_df.index
signals_df

In [ ]:
signals_df = signals_df.reindex(merged_df.index)

### Positions

In [ ]:
position_df = pd.DataFrame(index=signals_df.index)

for col in signals_df.columns:
    
    current_position = 0
    positions = []
    
    for signal in signals_df[col]:
        
        if signal == 1:
            current_position = 1
            
        elif signal == -1:
            current_position = -1
            
        elif signal == 0:
            current_position = 0  # EXIT must reset
        
        positions.append(current_position)
    
    position_df[col] = positions

In [ ]:
returns_df = merged_df.pct_change().fillna(0)

In [ ]:
position_df

### PnL

In [ ]:
pnl_df = pd.DataFrame(index=position_df.index)

In [ ]:
for col in position_df.columns:
    
    s1, s2 = col.split("_")
    beta_val = hedge_ratio[s1][s2]
    pnl_df[col] = position_df[col] * (returns_df[s1] - beta_val * returns_df[s2])

pnl_df

### Considering all the cointegrated pairs 

In [ ]:
portfolio_returns = pnl_df.mean(axis=1)
portfolio_returns

In [ ]:
cumulative_returns = (1 + portfolio_returns).cumprod()
cumulative_returns

In [ ]:
sharpe = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(252)
print("Sharpe:", sharpe)

In [ ]:
cum_max = cumulative_returns.cummax()
drawdown = cumulative_returns / cum_max - 1

max_dd = drawdown.min()
print("Max Drawdown:", max_dd)



In [ ]:
total_return = cumulative_returns.iloc[-1] - 1
print("Total Return:", total_return)


### Considering only top 5 pairs with the lowest half life

In [ ]:
top_five_pairs = cointegrated_pairs_df.sort_values(by="Half_Life")[:5]
top_five_pairs

In [ ]:
pairs_list = []
for idx, row in top_five_pairs.iterrows():
    pairs_str = f'{row["Stock1"]}_{row["Stock2"]}'
    pairs_list.append(pairs_str)
pairs_list

In [ ]:
top_five_portfolio_returns = pnl_df[pairs_list].mean(axis=1)
top_five_portfolio_returns

In [ ]:
cumulative_returns = (1 + portfolio_returns).cumprod()
cumulative_returns

In [ ]:
sharpe = (portfolio_returns.mean() / portfolio_returns.std()) * np.sqrt(252)
print("Sharpe:", sharpe)

In [ ]:
cum_max = cumulative_returns.cummax()
drawdown = cumulative_returns / cum_max - 1

max_dd = drawdown.min()
print("Max Drawdown:", max_dd)



In [ ]:
total_return = cumulative_returns.iloc[-1] - 1
print("Total Return:", total_return)


### Returns Data

In [ ]:
training_returns = np.log(train_df / train_df.shift(1)).fillna(0)
training_returns